# memory-hygiene: measure and repair drift in an agent memory store

Agent memory stores fail quietly. A fact gets written, it is later superseded, and nothing invalidates the old record. Both versions then sit in the store looking equally retrievable, so the agent can answer from either one. `memory-hygiene` is a pure standard library Python package that measures that drift and repairs it. It ships two entry points: `audit`, which is read-only and scores store health, and `maintain`, which is an idempotent scheduled repair pass that runs in dry-run mode by default.

Disclaimer: This code is provided "as-is" as a demonstration only to illustrate a potential solution. The code does not constitute a Google product or service of any kind, and Google offers no support, warranties, or liability of any kind with its regard. Whoever chooses to use this code accepts all responsibility related to it, including for its implementation, use, and ongoing maintenance. For the avoidance of doubt, this code is not eligible for the Google Open Source Software Vulnerability Rewards Program.

## What this notebook does

* Pulls the repo and confirms that the core package needs no install.
* Explains the four drift mechanisms the package detects.
* Builds a small synthetic store that deliberately contains all four problems.
* Runs `audit_store` on it and reads the report.
* Runs `run_maintenance` in dry-run mode, then with apply, and re-audits to see the health score move.
* Runs the seeded 30-day two-arm drift simulation.
* Shows, without executing, how to point the same audit at Agent Platform Memory Bank.

In [ ]:
# The core package is pure standard library. There is nothing to pip install
# for the audit and maintain paths. Cloning the repo is enough.
!git clone https://github.com/reute-oss/memory-hygiene.git
%cd memory-hygiene

## The four drift mechanisms

1. **Stale facts.** A record that was true when written and has aged past the point where it can still be trusted, with nothing marking it as expired. The audit calls these `ttl_overdue`: a still-open record whose `fact_class` TTL has already elapsed.
2. **Entity contradictions.** Two or more live records that assert incompatible values for the same attribute of the same entity.
3. **Duplicate drift.** Near identical records written at different times, so retrieval returns several copies of one fact and their small differences look like signal.
4. **Orphans.** Records that have never been read back since they were written. The package has no notion of a deleted owning entity and does not track one. A record is an orphan when its `retrieval_count` is 0, its `last_retrieved_at` is unset, and it is older than `orphan_grace_days`, which defaults to 14. Nothing ever retrieves them, so they cost storage and prompt budget and return nothing.

A fifth signal, `unbounded`, is reported alongside these. It counts open-ended records whose fact class has no TTL at all, so they can never expire on their own. It is a risk, not a defect, which is why it carries the smallest weight in the health score.

In [ ]:
# SYNTHETIC DEMONSTRATION DATA. These records are handwritten to contain all
# four drift mechanisms on purpose. Nothing here is a real measurement and the
# numbers the audit prints below describe only this toy store.
import datetime as dt

from memory_hygiene_store import JsonMemoryStore, MemoryRecord

now = dt.datetime(2026, 1, 15, 12, 0, 0, tzinfo=dt.timezone.utc)


def ago(days):
    return now - dt.timedelta(days=days)


records = [
    # Contradiction: two identity records on the same (entity, relation), both
    # still believed true, with values too dissimilar to be a rewording.
    MemoryRecord(
        id="r001",
        entity="user:ada",
        relation="employer",
        value="Globex",
        written_at=ago(200),
        fact_class="identity",
        last_retrieved_at=ago(2),
        retrieval_count=12,
        source_session="s-001",
    ),
    MemoryRecord(
        id="r002",
        entity="user:ada",
        relation="employer",
        value="Initech",
        written_at=ago(10),
        fact_class="identity",
        last_retrieved_at=ago(1),
        retrieval_count=3,
        source_session="s-014",
    ),
    # Duplicate drift: the same preference written twice, reworded.
    MemoryRecord(
        id="r003",
        entity="user:ada",
        relation="editor_theme",
        value="prefers a dark theme in the editor",
        written_at=ago(40),
        fact_class="preference",
        last_retrieved_at=ago(3),
        retrieval_count=5,
        source_session="s-004",
    ),
    MemoryRecord(
        id="r004",
        entity="user:ada",
        relation="editor_theme",
        value="Prefers a dark theme in the editor.",
        written_at=ago(35),
        fact_class="preference",
        last_retrieved_at=ago(4),
        retrieval_count=2,
        source_session="s-006",
    ),
    # Stale fact: an ephemeral record 30 days old against a 7 day TTL, still
    # open. It is read often, so it is overdue rather than an orphan.
    MemoryRecord(
        id="r005",
        entity="user:ada",
        relation="current_task",
        value="reviewing the Q4 planning memo",
        written_at=ago(30),
        fact_class="ephemeral",
        last_retrieved_at=ago(20),
        retrieval_count=4,
        source_session="s-008",
    ),
    # Orphan: written once, never retrieved, far past the 14 day grace period.
    MemoryRecord(
        id="r006",
        entity="user:ada",
        relation="signup_channel",
        value="signed up through the partner portal",
        written_at=ago(120),
        fact_class="preference",
        last_retrieved_at=None,
        retrieval_count=0,
        source_session="s-002",
    ),
    # A healthy record, so the store is not uniformly broken.
    MemoryRecord(
        id="r007",
        entity="user:ada",
        relation="seat_tier",
        value="enterprise",
        written_at=ago(10),
        fact_class="status",
        last_retrieved_at=ago(1),
        retrieval_count=7,
        source_session="s-013",
    ),
    # Already closed, so it is excluded from every believed-true rate.
    MemoryRecord(
        id="r008",
        entity="user:ada",
        relation="city",
        value="Haifa",
        written_at=ago(300),
        valid_to=ago(200),
        fact_class="status",
        last_retrieved_at=ago(210),
        retrieval_count=2,
        source_session="s-000",
    ),
    # Its live successor, 200 days old against a 90 day status TTL.
    MemoryRecord(
        id="r009",
        entity="user:ada",
        relation="city",
        value="Tel Aviv",
        written_at=ago(200),
        fact_class="status",
        last_retrieved_at=ago(2),
        retrieval_count=9,
        source_session="s-003",
    ),
]

store = JsonMemoryStore(path="demo_store.json", write_through=False)

# Start from scratch, so re-running this cell does not stack copies.
for existing in store.all_records():
    store.delete(existing.id)

store.add_many(records)
store.save()

print("records in synthetic store:", len(store.all_records()))
print("believed true right now:  ", len(store.live_records(now)))

In [ ]:
from memory_hygiene_audit import AuditConfig, audit_store

config = AuditConfig(
    now=now,
    duplicate_threshold=0.90,
    orphan_grace_days=14.0,
    ttl_policy={
        "ephemeral": dt.timedelta(days=7),
        "status": dt.timedelta(days=90),
        "preference": None,
        "identity": None,
    },
)

report = audit_store(store, config)
print(report.render())

## How to read the audit output

`report.render()` prints one block of text with four sections.

* **Header.** The timestamp the audit was evaluated at, the record count split into total, believed true and closed, and the two thresholds in force: `duplicate_threshold` and `orphan_grace_days`.
* **Findings.** One line per mechanism, each a count plus that count as a share of the believed-true records: contradiction pairs, near-duplicate clusters, ttl overdue, orphans never read, and unbounded records with no TTL. The rendered text carries counts only, not ids. The ids live on the report object, in `report.contradiction_pairs`, `report.contradicting_record_ids`, `report.duplicate_clusters`, `report.redundant_duplicate_ids`, `report.ttl_overdue_ids`, `report.orphan_ids` and `report.unbounded_ids`. `report.to_dict()` hands you the whole report as JSON for a dashboard or a CI gate.
* **Age distribution and staleness by fact class.** How old the believed-true records are, bucketed, and then per fact class the live count, the class TTL, mean and max age in days, and the share already overdue.
* **Memory health score.** One number out of 100, printed with its formula and a row per component giving weight, rate and penalty, so you can recompute the total by hand or change `HEALTH_WEIGHTS` and disagree with it.

Two things the report does not give you. There is no suggested action attached to a finding: deciding what to repair belongs to `maintain`, and the dry run in the next cell is where you see the proposed changes. And there is no reason string per record; the mechanism a record was grouped under is the reason.

A record can be counted by more than one mechanism. An unbounded record can also be an orphan. Contradictions and duplicates are the one exclusive pair: two records sharing an `(entity, relation)` with different values are a contradiction only when the values are dissimilar, and a near-duplicate when they are not. `duplicate_threshold` is the single dial that decides which.

If the counts look wrong, the cause is usually a threshold rather than a bug. `duplicate_threshold`, `orphan_grace_days` and `ttl_policy` are the three knobs on `AuditConfig` that move results the most.

In [ ]:
from memory_hygiene_maintain import MaintenancePolicy, run_maintenance

policy = MaintenancePolicy(
    now=now,
    ttl_policy=config.ttl_policy,
    duplicate_threshold=0.90,
    orphan_grace_days=14.0,
    close_superseded=True,
    merge_duplicates=True,
    apply_ttl=True,
)

# dry_run=True is the default. Nothing is written. This prints the plan only.
plan = run_maintenance(store, policy, dry_run=True)
print(plan.render())

In [ ]:
before = audit_store(store, config)

# Now actually apply the plan. run_maintenance is idempotent, so running it a
# second time on an already clean store should find nothing to do.
result = run_maintenance(store, policy, dry_run=False)
print(result.render())

after = audit_store(store, config)

print()
print("health score before:", round(before.health_score, 1))
print("health score after: ", round(after.health_score, 1))
print("records remaining:  ", len(store.all_records()))
print("still believed true:", len(store.live_records(now)))

# Idempotence check: a second pass should plan zero changes.
second_pass = run_maintenance(store, policy, dry_run=True)
print("second pass changes:", second_pass.change_count)

## The 30-day drift simulation

`demo_drift.py` runs a seeded, reproducible two-arm simulation over 30 simulated days. Both arms receive exactly the same stream of memory writes, updates and deletions. The control arm never runs maintenance. The treatment arm runs `maintain` on a schedule, the way you would in production.

The random seed is fixed in the script, so the run below produces the same numbers every time on any machine. Nothing is sampled from the network and nothing depends on the wall clock. If you want a different trajectory, change the seed at the top of the script and rerun.

In [ ]:
!python3 demo_drift.py

## Running against Agent Platform Memory Bank

Everything above runs against `JsonMemoryStore`, which is local and needs nothing installed. The same audit and maintenance logic runs against a real Agent Platform Memory Bank through the `VertexMemoryBankStore` adapter.

That adapter is the one part of the package that is not pure standard library, because it needs the Google Cloud client libraries. It lives behind an optional extra, so the core package stays dependency free. Nothing is published to PyPI, so install the extra from your clone of the repo:

```
pip install -e ".[vertex]"
```

`VertexMemoryBankStore` implements the same `MemoryStore` ABC as `JsonMemoryStore`, so `audit_store` and `run_maintenance` take it without any change. `open_store` can also build it from a URI if you prefer configuration over code.

In [ ]:
# NOT RUN IN THIS NOTEBOOK. This cell requires real GCP credentials, a real
# Agent Engine instance, and the optional vertex extra installed. It is here to
# show the shape of the call, and it has not been executed above.

from memory_hygiene_vertex import VertexMemoryBankStore
from memory_hygiene_store import utc_now
from memory_hygiene_audit import AuditConfig, audit_store
from memory_hygiene_maintain import MaintenancePolicy, run_maintenance

vertex_store = VertexMemoryBankStore(
    project="my-gcp-project",
    location="us-central1",
    agent_engine_id="1234567890123456789",
)

vertex_now = utc_now()

# audit_store is read-only. Running it against a live Memory Bank does not
# modify anything.
vertex_report = audit_store(vertex_store, AuditConfig(now=vertex_now))
print(vertex_report.render())

# And the repair pass, still in dry-run mode, which is the default.
vertex_plan = run_maintenance(
    vertex_store,
    MaintenancePolicy(now=vertex_now),
    dry_run=True,
)
print(vertex_plan.render())

### Honest status of the Memory Bank adapter

`VertexMemoryBankStore` was verified against a live Memory Bank on 7 October 2026, on one run with google-cloud-aiplatform 2.4.0. A full round trip passed on both
construction paths: write, read back every field, semantic retrieve, audit, update
and delete, finishing with an empty store.

Worth knowing how that went, because it is the point of this package. The first
release of this adapter could not write to Memory Bank at all. It passed metadata as
a top-level argument, and the real API nests it inside `config` as typed values. The
fake transport used in the unit tests accepted any argument, so 35 tests passed
against a call shape the live service rejects outright. The fake is now strict: it
mirrors the real signatures and refuses unknown arguments, and running the old
adapter against it reproduces the exact error the live service gave.

Scope of the verification: one run, one SDK version, `google-cloud-aiplatform`
2.4.0, on the `vertexai` surface. That surface is deprecated in favour of
`agentplatform`. The migration to the new client has NOT been verified live yet.


## Repo

Source, tests and issues: [https://github.com/reute-oss/memory-hygiene](https://github.com/reute-oss/memory-hygiene)